# 生成模拟数据

In [ ]:
import numpy as np
import pandas as pd


def generate_mock_portfolio_data(
    start_date="2024-01-02",
    periods=126,              # 半年大约 126 个交易日
    n_symbols=100,
    seed=42,
):
    rng = np.random.default_rng(seed)

    # 1) 交易日
    trade_dates = pd.bdate_range(start=start_date, periods=periods)

    # 2) 标的代码
    symbols = [f"STK{str(i).zfill(3)}" for i in range(1, n_symbols + 1)]

    # -------------------------------------------------
    # 生成 holdings_df: 每个交易日、每个标的一个权重
    # 让每天权重和为 1，且有一定稳定性
    # -------------------------------------------------
    holdings_records = []

    # 初始权重：Dirichlet 保证和为 1
    current_weight = rng.dirichlet(np.ones(n_symbols) * 3.0)

    for dt in trade_dates:
        # 每天加入一点扰动，再重新归一化
        noise = rng.normal(loc=0.0, scale=0.002, size=n_symbols)
        raw = current_weight + noise
        raw = np.clip(raw, 0.0001, None)   # 防止出现负权重
        current_weight = raw / raw.sum()

        for sym, w in zip(symbols, current_weight):
            holdings_records.append((dt, sym, float(w)))

    holdings_df = pd.DataFrame(
        holdings_records,
        columns=["trade_date", "symbol", "weight"]
    )

    # -------------------------------------------------
    # 生成 kline_df: 每个交易日、每个标的一个 close
    # 用随机收益率路径生成价格
    # -------------------------------------------------
    # 给每个标的一个不同的初始价格、漂移、波动率
    init_prices = rng.uniform(20, 200, size=n_symbols)
    mus = rng.normal(loc=0.0003, scale=0.00015, size=n_symbols)      # 日均收益漂移
    sigmas = rng.uniform(0.01, 0.03, size=n_symbols)                 # 日波动

    # 市场公共因子
    market_factor = rng.normal(loc=0.0002, scale=0.008, size=len(trade_dates))

    kline_records = []

    for j, sym in enumerate(symbols):
        prices = [init_prices[j]]
        beta = rng.uniform(0.8, 1.2)

        for t in range(1, len(trade_dates)):
            idio = rng.normal(loc=0.0, scale=sigmas[j])
            ret = mus[j] + beta * market_factor[t] + idio
            next_price = prices[-1] * (1.0 + ret)
            next_price = max(next_price, 1.0)  # 防止价格跌到 0 以下
            prices.append(next_price)

        for dt, px in zip(trade_dates, prices):
            kline_records.append((dt, sym, float(round(px, 4))))

    kline_df = pd.DataFrame(
        kline_records,
        columns=["trade_date", "symbol", "close"]
    )
    kline_df['open'], kline_df['high'], kline_df['low'] = kline_df['close'], kline_df['close'], kline_df['close']

    return holdings_df, kline_df


if __name__ == "__main__":
    holdings_df, kline_df = generate_mock_portfolio_data()

    print("=== holdings_df ===")
    print(holdings_df.head(10))
    print()
    print(holdings_df.shape)

    print("\n=== kline_df ===")
    print(kline_df.head(10))
    print()
    print(kline_df.shape)

    # 检查：每天权重是否约等于 1
    daily_weight_sum = holdings_df.groupby("trade_date")["weight"].sum()
    print("\n=== daily weight sum ===")
    print((daily_weight_sum - 1.0).abs().sort_values(ascending=False).head())  # 打印权重和不为 1 的日期
    holdings_df.to_csv("mock_holdings.csv", index=False)
    kline_df.to_csv("mock_kline.csv", index=False)